# NLP Lab 2 Evaluation: Advanced Text Representation & N-Gram Models
Welcome to your Lab 2 assessment! This notebook evaluates your conceptual understanding of text representation and language modeling by presenting advanced, real-world variations of the pipeline built during the lab session.

**Instructions:**
* Find the `### YOUR CODE HERE ###` markers in each block.
* Replace the `raise NotImplementedError()` with your functional code.
* Run the test cells below each exercise to verify your implementation. Do not edit the test cells!

## Extra: More grammatical sentence generation
This notebook also includes a small example that uses boundary markers, smoothing, and probability-based next-word selection to generate sentences that are more coherent than random sampling.

In [4]:
# Run this cell to import the necessary libraries
import re
import random
import nltk
import numpy as np
from collections import defaultdict, Counter

nltk.download('punkt', quiet=True)
from nltk.tokenize import sent_tokenize, word_tokenize

## Exercise 1: Sentence-Level Boundary Tagging
In N-gram modeling, we cannot let transitions cross randomly from the end of one sentence to the start of another. We must handle sentence boundaries explicitly.

Your task is to write a preprocessor that:
1. Splits a paragraph into individual sentences using NLTK's `sent_tokenize`.
2. For each sentence, converts words to lowercase and strips non-alphabetic characters.
3. Tokenizes the words.
4. Appends a start marker `<s>` to the beginning and an end marker `</s>` to the end of each sentence.
5. Returns a single **flattened** list of all processed tokens.

*Example Input:* `"Hello world! NLP is fun."`  
*Example Output:* `['<s>', 'hello', 'world', '</s>', '<s>', 'nlp', 'is', 'fun', '</s>']`

In [5]:
def preprocess_with_boundaries(text):
    """
    Splits text into sentences, cleans and tokenizes them, and adds sentence boundary markers.
    """
    flattened_tokens = []
    
    # 1. Split the raw text into sentences using NLTK's sent_tokenize
    ### YOUR CODE HERE ###
    
    sentences = sent_tokenize(text)
    
    for sentence in sentences:
        # Clean the sentence: lowercase and keep only alphabetic characters/spaces
        cleaned_sent = sentence.lower()
        cleaned_sent = re.sub(r'[^a-z\s]', '', cleaned_sent)
        
        # 2. Tokenize the cleaned sentence into words
        ### YOUR CODE HERE ###

        
        words = word_tokenize(cleaned_sent)
        
        # Only process if there are valid words in the sentence
        if len(words) > 0:
            # 3. Construct a new list containing '<s>', the words, and '</s>'
            # and extend the flattened_tokens list with it.
            ### YOUR CODE HERE ###
            flattened_tokens.append('<s>')
            for w in words:
                flattened_tokens.append(w)
            flattened_tokens.append('</s>')
            
    return flattened_tokens

In [6]:
# TEST CELL - DO NOT MODIFY
test_paragraph = "The cat sat. Dog ran!"
expected_result = ['<s>', 'the', 'cat', 'sat', '</s>', '<s>', 'dog', 'ran', '</s>']

student_result = preprocess_with_boundaries(test_paragraph)
assert student_result == expected_result, f"Expected {expected_result}, but got {student_result}"
print("Exercise 1 Passed! 🎉")

Exercise 1 Passed! 🎉


## Exercise 2: Manual Cosine Similarity Vector Comparison
In the lab, you learned how documents are mapped to numerical vectors using the Bag of Words (BoW) framework. Now, you will write the mathematical function to compare them.

The similarity of two documents represented by vectors $\mathbf{A}$ and $\mathbf{B}$ is calculated using **Cosine Similarity**:

$$\text{Cosine Similarity}(\mathbf{A}, \mathbf{B}) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$$

Where:
* $\mathbf{A} \cdot \mathbf{B}$ is the vector dot product.
* $\|\mathbf{A}\|$ is the Euclidean norm (magnitude) of vector $\mathbf{A}$.

Implement this function manually using NumPy without using any Scikit-Learn similarity metrics.

In [7]:
def cosine_similarity(vector_a, vector_b):
    """
    Computes the cosine similarity between two 1D numpy arrays.
    Should handle division by zero safely (return 0.0 if a vector has a norm of 0).
    """
    # 1. Compute the dot product between vector_a and vector_b
    dot_product = np.dot(vector_a, vector_b)
    
    # 2. Compute the Euclidean norm of both vectors using np.linalg.norm
    norm_a = np.linalg.norm(vector_a)
    norm_b = np.linalg.norm(vector_b)
    
    # 3. Calculate cosine similarity. Guard against division by zero!
    if norm_a == 0 or norm_b == 0:
        return 0.0
    
    similarity = dot_product / (norm_a * norm_b)
    return similarity

In [8]:
# TEST CELL - DO NOT MODIFY
vec_a = np.array([1, 0, 1, 1])
vec_b = np.array([1, 1, 0, 1])

# Expected: Dot product is 2. Norms are sqrt(3) and sqrt(3). Similarity is 2/3 ≈ 0.6667
sim = cosine_similarity(vec_a, vec_b)
assert round(sim, 4) == 0.6667, f"Expected 0.6667, but got {round(sim, 4)}"

# Zero vector test
assert cosine_similarity(vec_a, np.array([0, 0, 0, 0])) == 0.0, "Should return 0.0 when comparing with a zero vector."
print("Exercise 2 Passed! 🎉")

Exercise 2 Passed! 🎉


## Exercise 3: Add-One (Laplace) Smoothing for Bigrams
A severe limitation of raw Maximum Likelihood Estimation (MLE) is the "zero-probability problem"—if a bigram sequence was never seen in training, its probability is $0.0$. 

To fix this, we apply **Laplace (Add-One) Smoothing**. The formula for the smoothed probability of word $w_i$ given its predecessor $w_{i-1}$ is:

$$P_{\text{Laplace}}(w_i \mid w_{i-1}) = \frac{C(w_{i-1}, w_i) + 1}{C(w_{i-1}) + |V|}$$

Where:
* $C(w_{i-1}, w_i)$ is the raw bigram count.
* $C(w_{i-1})$ is the unigram count of the history word.
* $|V|$ is the size of the unique vocabulary (including boundary tags) in the corpus.

Implement a smoothed bigram probability compiler.

In [9]:
def build_smoothed_bigram_model(tokens):
    """
    Builds a bigram model with Laplace (Add-One) Smoothing.
    """
    unique_vocab = set(tokens)
    V_size = len(unique_vocab)
    
    unigram_counts = Counter(tokens)
    bigram_counts = defaultdict(Counter)
    
    for i in range(len(tokens) - 1):
        w1, w2 = tokens[i], tokens[i+1]
        bigram_counts[w1][w2] += 1
    
    smoothed_model = defaultdict(dict)
    
    for w1 in unique_vocab:
        history = (w1,)
        for w2 in unique_vocab:
            bigram = bigram_counts[w1][w2]
            unigram = unigram_counts[w1]
            p_smoothed = (bigram + 1) / (unigram + V_size)
            smoothed_model[history][w2] = p_smoothed
            
    return smoothed_model

In [10]:
# Extra example: more grammatical sentence generation
import random
import re
from collections import defaultdict, Counter
from nltk.tokenize import sent_tokenize, word_tokenize


def preprocess_with_boundaries(text):
    flattened_tokens = []
    sentences = sent_tokenize(text)
    for sentence in sentences:
        cleaned_sent = sentence.lower()
        cleaned_sent = re.sub(r'[^a-z\s]', '', cleaned_sent)
        words = word_tokenize(cleaned_sent)
        if len(words) > 0:
            flattened_tokens.append('<s>')
            for w in words:
                flattened_tokens.append(w)
            flattened_tokens.append('</s>')
    return flattened_tokens


def build_smoothed_bigram_model(tokens):
    model = defaultdict(Counter)
    for i in range(len(tokens) - 1):
        history = (tokens[i],)
        next_word = tokens[i + 1]
        model[history][next_word] += 1

    for history, counter in model.items():
        total = sum(counter.values())
        for word, count in list(counter.items()):
            counter[word] = count / total
    return model


sample_text = "The cat sat on the mat. The dog ran quickly. The bird sang in the morning."
sample_tokens = preprocess_with_boundaries(sample_text)

# Build a trigram model with probabilities.
trigram_model = defaultdict(Counter)
for i in range(len(sample_tokens) - 2):
    history = tuple(sample_tokens[i:i+2])
    next_word = sample_tokens[i+2]
    trigram_model[history][next_word] += 1

for history, counter in trigram_model.items():
    total = sum(counter.values())
    for word, count in counter.items():
        counter[word] = count / total

# Build a smoothed bigram model for backoff.
bigram_model = build_smoothed_bigram_model(sample_tokens)


def generate_more_grammatical_sentence(trigram_model, bigram_model, n=3, max_length=12, temperature=0.2):
    sentence = ["<s>", "<s>"]

    while len(sentence) < max_length:
        history = tuple(sentence[-(n-1):])

        if len(history) == 2 and history in trigram_model:
            candidates = trigram_model[history]
        else:
            bigram_history = (history[-1],)
            candidates = bigram_model.get(bigram_history, {})

        if not candidates:
            break

        sorted_words = sorted(candidates.items(), key=lambda x: x[1], reverse=True)

        if random.random() > temperature:
            next_word = sorted_words[0][0]
        else:
            words = [w for w, _ in sorted_words]
            probs = [p for _, p in sorted_words]
            next_word = random.choices(words, weights=probs, k=1)[0]

        if next_word == "</s>":
            break

        sentence.append(next_word)

    words = sentence[2:]
    if not words:
        return ""

    text = " ".join(words)
    if text:
        text = text[0].upper() + text[1:]
    if not text.endswith("."):
        text += "."
    return text

for _ in range(5):
    print(generate_more_grammatical_sentence(trigram_model, bigram_model))

The cat sat on the mat.
The cat sat on the mat.
The cat sat on the mat.
The cat sat on the mat.
The cat sat on the mat.


## Exercise 4: The Shannon Backoff Predictor
When simulating Shannon's Guessing Game, a high-order model (like a Trigram) often fails to predict a word because the specific two-word history was never seen during training. 

To make our predictor robust, you will implement a **Backoff Predictor**:
1. It tries to search the Trigram model using the full history `(w_i-2, w_i-1)`.
2. If the Trigram history is **not** present in the trigram model, it "backs off" to search the Bigram model using the shortened history `(w_i-1,)`.
3. If the Bigram history is also missing, it returns an empty list `[]`.

This teaches you how cascading model structures work in real-world search auto-completes.

In [11]:
def backoff_predictor(trigram_history, trigram_model, bigram_model):
    """
    Attempts trigram lookup. Backs off to bigram if history is missing.
    
    Inputs:
        trigram_history: A tuple of 2 words (w_i-2, w_i-1)
        trigram_model: Dict of Trigram probabilities
        bigram_model: Dict of Bigram probabilities
    """
    # 1. Extract the bigram history (the second word as a single-element tuple)
    bigram_history = (trigram_history[1],)
    
    # 2. Check if the trigram_history exists in the trigram_model
    if trigram_history in trigram_model:
        candidates = trigram_model[trigram_history]
        return sorted(candidates.items(), key=lambda x: x[1], reverse=True)
    
    # 3. Else, check if the bigram_history exists in the bigram_model
    if bigram_history in bigram_model:
        candidates = bigram_model[bigram_history]
        return sorted(candidates.items(), key=lambda x: x[1], reverse=True)
    
    # 4. If neither exists, return an empty list []
    return []

In [12]:
# TEST CELL - DO NOT MODIFY
# Mock models for testing
mock_trigram = {
    ('i', 'am'): {'sam': 0.8, 'free': 0.2}
}
mock_bigram = {
    ('am',): {'sam': 0.5, 'free': 0.4, 'happy': 0.1},
    ('running',): {'fast': 1.0}
}

# Test 1: Exact Trigram Match
res1 = backoff_predictor(('i', 'am'), mock_trigram, mock_bigram)
assert res1[0][0] == 'sam' and len(res1) == 2, "Trigram lookup failed to find correct match."

# Test 2: Backoff to Bigram Match (History ('she', 'am') -> backs off to ('am',))
res2 = backoff_predictor(('she', 'am'), mock_trigram, mock_bigram)
assert len(res2) == 3 and res2[0][0] == 'sam', "Backoff to Bigram failed."

# Test 3: Total Miss
res3 = backoff_predictor(('we', 'are'), mock_trigram, mock_bigram)
assert res3 == [], "Should return an empty list when histories are entirely missing."

print("Exercise 4 Passed! 🎉")
print("\nCongratulations! You have successfully coded and passed all assessment modules.")

Exercise 4 Passed! 🎉

Congratulations! You have successfully coded and passed all assessment modules.
